# Coupled Continuous-Time Spectral Dynamics for One-Step Music Forecasting: A Controlled Feasibility Study

*Executable research protocol · Academic V2 · Google Colab*

## Abstract

Short-horizon prediction of music spectra is commonly treated as a discrete regression problem, although spectral transitions simultaneously exhibit local frequency structure and temporal dependencies that may be represented by continuous dynamics. This study investigates whether a coupled neural partial differential equation–ordinary differential equation (PDE–ODE) formulation offers a useful inductive bias for forecasting the next time–frequency observation. A one-dimensional discretized spectral field evolves through a learned convolutional reaction term and a nonnegative diffusion coefficient, while a low-dimensional neural ODE represents global temporal context. The two subsystems exchange information during differentiable numerical integration. We evaluate the complete architecture against two controlled ablations that remove diffusion or latent-state influence, together with a persistence predictor. Inputs comprise log-magnitude short-time Fourier transform features from a reproducible, music-like synthetic recording; training, validation, and test observations are partitioned chronologically, and normalization uses the training partition only. The primary endpoint is held-out one-step spectral mean squared error, complemented by analysis of spectral novelty and computational cost. We hypothesize that the coupled formulation may better capture transitions requiring both localized and global context, whereas excessive diffusion may suppress transient structure. This executable study is a feasibility protocol; it does not establish empirical superiority or generalization to unseen recordings before the experiments are run.

## 1. Introduction and research objective

The coupled neural PDE–ODE paradigm [1] provides a mechanism for jointly modelling spatially structured states and lower-dimensional temporal variables. However, translating this inductive bias to music spectrograms is nontrivial: adjacency between spectral bands is a representation choice, not a physical diffusion law. The empirical question is whether the additional continuous coupling improves prediction beyond simpler alternatives, rather than whether it more faithfully encodes acoustic physics.

**Research question (RQ1).** Does coupled spectral-field and latent-state evolution reduce one-step prediction error relative to removing either mechanism and to persistence, under a controlled training protocol?

**Contribution under evaluation.** The study implements an explicitly coupled, differentiable continuous-time spectral predictor and a two-factor ablation design. It is a controlled adaptation of prior coupled-dynamics work, not a claim of a previously unreported PDE governing music.

### 1.1. Unresolved problem: what is an appropriate continuous operator for music?

The present architecture imports the mathematical form of reaction–diffusion into log-magnitude frequency space. It is not known whether diffusion between adjacent frequency bins is an informative inductive bias, a convenient regularizer, or a source of harmful harmonic smoothing. Crucially, a reduction in average forecasting loss cannot resolve this distinction: a model can produce smooth, low-error spectra while attenuating note attacks and narrow harmonics.

**Critical question CQ1 — operator validity and identifiability.** Under which acoustic transformations can local spectral evolution and global context be identified separately from partial audio observations? Does a diffusion operator remain beneficial under changes of instrument, pitch register, or tempo?

**Research direction.** Compare the existing frequency-neighbor diffusion with (i) learned nonlocal harmonic coupling, (ii) diffusion-free convolution, and (iii) a complex-spectrum operator retaining phase. Introduce controlled instrument–tempo interventions and test whether the learned global state changes when rhythm changes but timbre is held fixed.

**Falsifiable criterion.** Any claimed improvement must survive instrument-disjoint evaluation and must not degrade event-level onset accuracy, high-frequency detail, or phase-sensitive audio reconstruction relative to strong neural baselines. The current one-recording experiment supplies only a *feasibility test*, not this identification result. Prior work on jointly modeling musical onsets and frames illustrates why onset preservation requires separate evaluation [4]; MAESTRO supplies a possible annotated benchmark [5].

## 2. Proposed method

Let $u(t)\in\mathbb{R}^{F}$ denote a normalized log-spectral vector over $F$ frequency bands and $z(t)\in\mathbb{R}^{d}$ a low-dimensional latent state. The implemented model jointly evolves

$$
\frac{du}{dt}=D_\theta \Delta_f u + R_\theta(u) + C_\theta(z),\qquad
\frac{dz}{dt}=G_\phi\!\bigl(z,\operatorname{mean}_f u\bigr),
$$

where $\Delta_f$ is a finite-difference operator with reflecting boundary conditions, $D_\theta=\operatorname{softplus}(d_\theta)\geq0$, $R_\theta$ is a convolutional reaction network, and $C_\theta$ maps latent state to spectral corrections. `torchdiffeq` integrates the coupled state and permits gradient-based training. The one-step prediction is $\hat u_{k+1}=\mathcal S_{\Delta t}(u_k,z_k;\theta,\phi)$, with $\mathcal S$ denoting numerical integration.

**Identification strategy.** The complete model is compared with (i) $D_\theta=0$ and (ii) suppressed latent coupling, using the same optimization schedule. A no-learning persistence baseline provides a necessary control for highly autocorrelated frames. The proposed coupled mechanism is highlighted in the architecture and evaluation figures, but graphical highlighting carries no implication of superior performance.

## 3. Expected outcomes and falsifiable hypotheses

**H1 (predictive utility).** Joint local/global dynamics reduce held-out one-step spectral error relative to at least one ablation when the synthetic sequence contains abrupt harmonic or percussive changes. **H0:** the full model does not reduce error relative to the strongest baseline. **H2 (failure mode).** A positive learned diffusion coefficient may increase high-frequency or onset-associated error through smoothing; the additional model capacity and numerical integration may also worsen cost efficiency.

These statements are **predictions made before evaluation**, not reported findings. The evidence will be the computed test MSE, novelty visualization, training/runtime data, and explicit comparisons with persistence. A single synthetic signal cannot support real-world superiority claims.

## 4. Experimental procedure

**Step 1 — Data representation.** Generate or upload a permitted WAV recording; compute a compact log-magnitude STFT and a chronological split with training-only normalization. **Step 2 — Model specification.** Implement the coupled spectral field and global latent evolution; construct the diffusion-off and global-state-off ablations. **Step 3 — Estimation.** Fit models using the same Adam schedule, selecting checkpoints with validation error. **Step 4 — Evaluation.** Compare the frozen predictions on held-out frames against the persistence baseline, plot spectral novelty, and retain negative outcomes. The executable cells below correspond to these numbered steps.


### Algorithm and experimental diagrams

The first diagram marks the coupled spectral dynamics as the **proposed mechanism**. The second separates that mechanism from benchmarks; the third makes validation/held-out-test boundaries explicit. These diagrams are generated by notebook code and saved as SVGs.

### Interpretation of the methodological figures

**Figure 1 — Mechanistic architecture:** the highlighted node identifies the *proposed scientific mechanism*, not a generic preprocessing step. The incoming arrows specify evidence or state needed for the method; outgoing arrows specify the predictions or edits being evaluated.

**Figure 2 — Comparative evaluation:** proposed-method results are distinguished from baselines using a thick outline or hatching. A visually highlighted method is **not** an assumed winner. Its score must be measured on the same held-out target as the baselines.

**Figure 3 — Data partitioning and verification:** training diagnostics may inform candidate generation; validation decides acceptance; the test partition is reserved for final reporting. Failed model downloads, invalid outputs, and rejected repairs remain visible and are not replaced by fabricated scores.

These diagrams are executable Graphviz figures generated inside Colab; the paper-style explanations and executable experiments remain the primary deliverable.


In [ ]:
%pip -q install graphviz torchdiffeq soundfile scipy pandas matplotlib

In [ ]:
import os, math, time, random, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from torchdiffeq import odeint
from scipy.signal import stft, resample_poly
from scipy.io import wavfile
SEED=211100343
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device,'torch:',torch.__version__)


In [ ]:
# Three executable research diagrams, saved inside the Colab runtime.
# The thicker proposed-method border is intentionally distinct from baseline methods.
from IPython.display import display, SVG
from graphviz import Digraph

def research_diagram(name, nodes, edges, proposed, caption):
    dot=Digraph(name,format='svg')
    dot.attr(rankdir='LR',fontname='Helvetica',labelloc='t',label=caption,nodesep='0.35',ranksep='0.45')
    for key,label in nodes:
        if key in proposed:
            dot.node(key,label,shape='box',style='rounded,bold',color='#B45309',penwidth='3.5',
                     fillcolor='#FFF7ED',fontcolor='#7C2D12')
        else:
            dot.node(key,label,shape='box',style='rounded',color='#64748B')
    for u,v in edges: dot.edge(u,v)
    try:
        svg=dot.pipe(format='svg').decode('utf-8')
        display(SVG(svg));Path(name+'.svg').write_text(svg)
    except Exception as exc:
        print('Graphviz rendering unavailable:',exc,'\n',dot.source)

research_diagram('v2_method',
    [('audio','Audio / spectrogram'),('diagnose','Diagnostics / local errors'),
     ('local','Local PDE-inspired operator'),('global','Persistent or short-horizon ODE'),
     ('proposer','PROPOSED: model / weight proposer'),('solve','Differentiable integration'),
     ('verify','Independent verifier')],
    [('audio','diagnose'),('diagnose','local'),('diagnose','global'),
     ('local','proposer'),('global','proposer'),('proposer','solve'),('solve','verify')],
    ['local','global','proposer','solve'], 'METHOD: '+'Coupled spectral dynamics')
research_diagram('v2_evaluation',
    [('data','Recordings / synthetic signals'),('split','Train → validation → test'),
     ('proposed','PROPOSED: Coupled PDE–ODE'),('base','Classical / neural baselines'),
     ('hf','3 Hugging Face models (where applicable)'),('gate','Validation / selection gate'),
     ('out','Held-out MSE, onsets, latency')],
    [('data','split'),('split','proposed'),('split','base'),('split','hf'),
     ('proposed','gate'),('base','gate'),('hf','gate'),('gate','out')],
    ['proposed'], 'EVALUATION: isolate the proposed method from benchmarks')
research_diagram('v2_decisions',
    [('frozen','Frozen candidate / fixed design'),('probe','Residual / sensitivity probe'),
     ('action','PROPOSED: targeted intervention'),('validation','Disjoint validation'),
     ('accepted','Accept only if improvement'),('rejected','Reject / record failure'),
     ('test','One-shot held-out evaluation')],
    [('frozen','probe'),('probe','action'),('action','validation'),
     ('validation','accepted'),('validation','rejected'),('accepted','test'),('rejected','test')],
    ['action'], 'VERIFICATION: no candidate accesses test labels during selection')

## Step 1 — Construct controlled music-like observations

We generate a time-frequency signal with chords and percussive pulses or accept a permitted WAV. The output is a log-spectral field. This is a deliberately easy diagnostic setting, not a genre benchmark.

In [ ]:
SAMPLE_RATE=8000
DURATION=5.0
USE_UPLOAD=False
def synth_audio(seed=3):
    rng=np.random.default_rng(seed)
    t=np.arange(int(SAMPLE_RATE*DURATION))/SAMPLE_RATE
    x=np.zeros_like(t)
    chords=[[220,277.18,329.63],[196,246.94,293.66],[174.61,220,261.63],[164.81,207.65,246.94]]
    for j,chord in enumerate(chords):
        a,b=j*1.25,(j+1)*1.25
        idx=(t>=a)&(t<b)
        tau=t[idx]-a
        envelope=np.minimum(1,tau*18)*np.exp(-0.35*tau)
        for f in chord:
            x[idx]+=envelope*np.sin(2*np.pi*f*tau+rng.uniform(-.2,.2))/len(chord)
    for beat in np.arange(0,DURATION,.5):
        ix=int(beat*SAMPLE_RATE)
        length=min(500,len(x)-ix)
        if length>0:
            pulse=rng.standard_normal(length)*np.exp(-np.arange(length)/90)
            x[ix:ix+length]+=.09*pulse
    return (x/(max(1e-6,np.max(np.abs(x))))).astype(np.float32)
if USE_UPLOAD:
    from google.colab import files
    upload=files.upload()
    fname=next(iter(upload))
    sr,audio=wavfile.read(fname)
    if audio.ndim>1: audio=audio.astype(np.float32).mean(axis=1)
    audio=audio.astype(np.float32)
    audio/=max(1e-6,np.max(np.abs(audio)))
    if sr!=SAMPLE_RATE:
        from math import gcd
        d=gcd(int(sr),SAMPLE_RATE)
        audio=resample_poly(audio,SAMPLE_RATE//d,int(sr)//d)
    audio=audio[:int(DURATION*SAMPLE_RATE)]
else:
    audio=synth_audio()
print('Samples:',len(audio),'duration:',round(len(audio)/SAMPLE_RATE,2),'s')
freqs,times,Z=stft(audio,fs=SAMPLE_RATE,nperseg=256,noverlap=192,boundary=None)
spec=np.log1p(10*np.abs(Z)).astype(np.float32)
# 64 frequency bands, using first 128 FFT bins in pairs
spec=spec[:128].reshape(64,2,-1).mean(axis=1)
print('Spectrogram:',spec.shape)
fig,ax=plt.subplots(figsize=(11,3))
ax.imshow(spec,aspect='auto',origin='lower',extent=[times[0],times[-1],0,64])
ax.set(xlabel='Time (s)',ylabel='Frequency band',title='Log magnitude spectrogram')
plt.show()


## Step 2 — Propose and instantiate the continuous dynamics

We implement local convolutional reaction/diffusion and a compact neural ODE. Coupling is the **method under investigation**; ablations independently disable diffusion or the latent influence.

In [ ]:
BANDS=64
class CoupledDynamics(nn.Module):
    def __init__(self, mode='coupled', hidden=8):
        super().__init__()
        assert mode in ('coupled','no_diffusion','no_latent')
        self.mode=mode
        self.hidden=hidden
        self.diffusion_raw=nn.Parameter(torch.tensor(-2.0))
        self.local=nn.Sequential(nn.Conv1d(1,8,5,padding=2),nn.Tanh(),
                                 nn.Conv1d(8,1,3,padding=1))
        self.global_net=nn.Sequential(nn.Linear(hidden+1,16),nn.Tanh(),
                                      nn.Linear(16,hidden))
        self.inject=nn.Linear(hidden,1,bias=False)
    def forward(self,t,state):
        u,z=state
        # u: batch x bands, z: batch x hidden
        padded=torch.nn.functional.pad(u.unsqueeze(1),(1,1),mode='replicate').squeeze(1)
        lap=padded[:,:-2]-2*u+padded[:,2:]
        D=torch.nn.functional.softplus(self.diffusion_raw)
        if self.mode=='no_diffusion': D=D*0
        reaction=self.local(u.unsqueeze(1)).squeeze(1)
        if self.mode=='no_latent':
            dz=torch.zeros_like(z)
            coupling=torch.zeros_like(u)
        else:
            dz=self.global_net(torch.cat([z,u.mean(dim=-1,keepdim=True)],dim=-1))
            coupling=self.inject(z).expand_as(u)
        return D*lap+reaction+coupling,dz
    def step(self,u,dt):
        z=torch.zeros(u.shape[0],self.hidden,device=u.device,dtype=u.dtype)
        # Differentiable fixed-step integration of the coupled ODE
        t=torch.tensor([0.,float(dt)],device=u.device,dtype=u.dtype)
        traj=odeint(self,(u,z),t,method='rk4',options={'step_size':float(dt)/2})
        return traj[0][-1]


## Step 3 — Define splits and train the candidates

Normalization is fit only on the training portion. Adam fits the coupled and ablated models with the same schedule; validation selects the checkpoint. A persistence baseline prevents misleading conclusions from slow-changing frames.

In [ ]:
X=spec.T
# normalize on first 60% of chronological frames only
N=len(X); n_train=int(.6*N); n_val=int(.8*N)
mu=X[:n_train].mean(axis=0,keepdims=True)
sd=X[:n_train].std(axis=0,keepdims=True)+1e-3
Xn=np.clip((X-mu)/sd,-5,5).astype(np.float32)
def pairs(start,end):
    return (torch.tensor(Xn[start:end-1],device=device),
            torch.tensor(Xn[start+1:end],device=device))
train_x,train_y=pairs(0,n_train)
val_x,val_y=pairs(n_train,n_val)
test_x,test_y=pairs(n_val,N)
DT=float(np.median(np.diff(times)))
print('train/val/test pairs:',len(train_x),len(val_x),len(test_x),'dt:',DT)


In [ ]:
import copy
EPOCHS=35
BATCH=32
results={}
predictions={}
for mode in ('coupled','no_diffusion','no_latent'):
    torch.manual_seed(SEED)
    model=CoupledDynamics(mode=mode).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=2e-3)
    best=float('inf'); best_state=None
    started=time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        for ix in torch.randperm(len(train_x),device=device).split(BATCH):
            opt.zero_grad()
            pred=model.step(train_x[ix],DT)
            loss=(pred-train_y[ix]).square().mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            opt.step()
        model.eval()
        with torch.no_grad():
            val=float((model.step(val_x,DT)-val_y).square().mean())
        if val<best:
            best=val
            best_state=copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    with torch.no_grad():
        pred=model.step(test_x,DT)
        mse=float((pred-test_y).square().mean())
        mae=float((pred-test_y).abs().mean())
        predictions[mode]=pred.detach().cpu().numpy()
    results[mode]={'test_mse':mse,'test_mae':mae,'validation_mse':best,
                   'training_seconds':time.perf_counter()-started,
                   'diffusion':float(torch.nn.functional.softplus(model.diffusion_raw))}
    print(mode,results[mode])
results['persistence']={'test_mse':float((test_x-test_y).square().mean()),
                        'test_mae':float((test_x-test_y).abs().mean())}
print('persistence',results['persistence'])


## 5. Experimental evaluation — Proposed method versus ablations

The original outputs include spectral novelty and comparison bars. The additional V2 chart highlights **coupled** and prints whether it improves over persistence and both ablations. For publication, these single-track results should be treated as a pilot.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
names=list(results)
axes[0].bar(names,[results[k]['test_mse'] for k in names])
axes[0].tick_params(axis='x',rotation=20)
axes[0].set(ylabel='Held-out next-frame MSE ↓',title='Model comparison')
novelty=np.maximum(0,np.diff(spec,axis=1)).sum(axis=0)
axes[1].plot(times[1:],novelty)
axes[1].set(xlabel='Time (s)',ylabel='Positive spectral flux',title='Spectral novelty / onset proxy')
fig.tight_layout()
fig.savefig('music_pde_ode_results.png',dpi=160)
plt.show()
fig,ax=plt.subplots(1,3,figsize=(14,3))
true=test_y.detach().cpu().numpy().T
for a,(title,arr) in zip(ax,[('Reference',true),
                             ('Coupled PDE–ODE',predictions['coupled'].T),
                             ('No diffusion',predictions['no_diffusion'].T)]):
    a.imshow(arr,origin='lower',aspect='auto',vmin=-2,vmax=3)
    a.set_title(title);a.set_xlabel('Test frame')
ax[0].set_ylabel('Frequency band')
fig.tight_layout()
fig.savefig('music_pde_ode_spectrograms.png',dpi=160)
plt.show()
pd.DataFrame(results).T.to_csv('music_pde_ode_metrics.csv')
Path('music_pde_ode_metadata.json').write_text(json.dumps(
 {'seed':SEED,'sample_rate':SAMPLE_RATE,'epochs':EPOCHS,'bands':BANDS,
  'train_pairs':len(train_x),'validation_pairs':len(val_x),
  'test_pairs':len(test_x),'uploaded_audio':USE_UPLOAD,
  'warning':'One-signal proof of concept; no generalization claim'},indent=2))
print(pd.DataFrame(results).T.round(6))
print('Saved: music_pde_ode_results.png, music_pde_ode_spectrograms.png, '
      'music_pde_ode_metrics.csv, music_pde_ode_metadata.json')


In [ ]:
pilot_df=pd.DataFrame([{'model':k,'test_mse':v['test_mse']} for k,v in results.items()])
print(pilot_df.sort_values('test_mse').to_string(index=False))
if 'coupled' in results:
    print('Coupled beats all tested alternatives:',all(results['coupled']['test_mse']<v['test_mse'] for k,v in results.items() if k!='coupled'))

In [ ]:
# A publication-oriented plot: the evaluated proposal is visually distinct.
fig,ax=plt.subplots(figsize=(11,4.5))
plot_data=pilot_df
plot_names=plot_data['model'].astype(str).tolist()
plot_values=plot_data['test_mse'].astype(float).tolist()
bars=ax.bar(plot_names,plot_values,alpha=.64)
for label,bar in zip(plot_names,bars):
    if label=='coupled':
        bar.set_edgecolor('black');bar.set_linewidth(3);bar.set_hatch('///')
        ax.annotate('PROPOSED METHOD', (bar.get_x()+bar.get_width()/2,bar.get_height()),
                    xytext=(0,12),textcoords='offset points',ha='center',fontsize=8,weight='bold')
ax.set(ylabel='test_mse (lower is better)',title='Proposed coupled PDE–ODE versus simple baselines')
ax.tick_params(axis='x',rotation=32)
fig.tight_layout();fig.savefig('v2_basic_proposed_comparison.png',dpi=160);plt.show()

## 6. Interpretation of numerical results

After executing the notebook, use the reported held-out metrics to determine whether the proposed coupled dynamics outperform the ablations and simple baselines. A win on one synthetic signal is **not** evidence of real-music generalization. If the proposed model loses, preserve that negative result and investigate diffusion strength, temporal context, and baseline capacity.

**Future research:** repeat across independent recordings and random seeds, compare against three pretrained Hugging Face forecasting checkpoints in the recording-level V2 notebook, and evaluate persistent state and targeted weight-space repair separately.


## Research agenda: evidentiary standards

The research questions above are intentionally harder than the experiments currently executed by this notebook. **A hypothesis is not a measured result**, and an unimplemented extension is not a contribution. The diagrams identify the proposed scientific mechanism; they must not visually encode superiority that the numerical evidence has not established.

A claim of superiority requires a preregistered outcome, independent train/validation/test groups, repeated random seeds, uncertainty estimates, robust baselines, and numerical/compute fairness. When comparing pretrained Hugging Face models, distinguish time-series forecasting checkpoints from instruction models proposing weight edits. Unavailable checkpoints and rejected repairs remain explicitly missing/rejected, never assigned fabricated performance.

The highest-impact unresolved questions are (i) separating smooth dynamics from abrupt musical events, (ii) identifying transferable local/global representations, (iii) preserving harmonic and phase detail, (iv) maintaining long-range structure, and (v) performing non-regressive, localized neural weight-space repair under distribution shift. The present notebook provides an initial component of that program and specifies which claims remain future work.


## 7. Discussion and limitations

**Discussion.** The experiment tests whether a coupled neural PDE–ODE is a useful modelling bias for one-step spectral forecasting. Any observed MSE difference must be interpreted in light of limited synthetic diversity, imperfect capacity matching, and the fact that spectral diffusion is not a known physical law for musical signals. The ablations support analysis of two components, but they do not establish causality across unseen recordings or musical domains.

## 8. Conclusion and future research

**Conclusion.** This notebook provides an executable test of the coupled-dynamics hypothesis; its scientific conclusion is conditional on the observed held-out results. No improvement is asserted in advance. In particular, an apparent gain over one ablation but not over persistence would not demonstrate practical forecasting value.

**Future research.** Evaluate recording- and artist-disjoint real-audio datasets; maintain persistent latent state for longer rollouts; use stronger GRU, convolutional, Transformer, and pretrained forecasting baselines; quantify onset and high-frequency errors; and investigate solver sensitivity, parameter-matched controls, and JAX-based simulation. The recording-level V2 study implements several of these extensions.

## References

[1] M. Habiba and B. A. Pearlmutter, *Continuous Convolutional Neural Networks: Coupled Neural PDE and ODE* (2021). https://arxiv.org/abs/2111.00343

[2] R. T. Q. Chen, Y. Rubanova, J. Bettencourt, and D. Duvenaud, *Neural Ordinary Differential Equations*, NeurIPS (2018). https://arxiv.org/abs/1806.07366

[3] R. T. Q. Chen, `torchdiffeq`, differentiable ODE solvers. https://github.com/rtqichen/torchdiffeq

**Reproducibility and licensing.** The default demonstration uses a synthetic recording; user-uploaded audio must be licensed or otherwise permitted for analysis. Model initialization and splits are controlled by a seed, but a single run is not a statistical comparison.
[4] C. Hawthorne et al., *Onsets and Frames: Dual-Objective Piano Transcription*, ISMIR (2018). https://arxiv.org/abs/1710.11153

[5] C. Hawthorne et al., *Enabling Factorized Piano Music Modeling and Generation with the MAESTRO Dataset*, ICLR (2019). https://arxiv.org/abs/1810.12247
